In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 10 — Local Code Explainer
## Explain Code Snippets and Detect Simple Issues

**Stack:** Ollama · LangChain · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

code_samples = {
    "binary_search": """
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
""",
    "buggy_function": """
def calculate_average(numbers):
    total = 0
    for i in range(len(numbers)):
        total += numbers[i]
    average = total / len(numbers)
    return average

result = calculate_average([])
""",
    "complex_decorator": """
import functools
import time

def retry(max_attempts=3, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
                    time.sleep(delay * (2 ** attempt))
            return None
        return wrapper
    return decorator
""",
}
print(f"Loaded {len(code_samples)} code samples")

## Step 2 — Line-by-Line Explanation

In [ ]:
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a patient code tutor. Explain code line by line:
- Explain what each line does in plain English
- Note the algorithm or pattern being used
- Explain time/space complexity
- Use analogies for complex concepts
- Target audience: intermediate Python developer"""),
    ("human", "Explain this code line by line:\n```python\n{code}\n```")
])

explain_chain = explain_prompt | llm | StrOutputParser()

for name, sample in code_samples.items():
    print(f"\n{'='*60}")
    print(f"Code: {name}")
    print('='*60)
    explanation = explain_chain.invoke({"code": sample})
    print(explanation)

## Step 3 — Bug Detection

In [ ]:
class Bug(BaseModel):
    line_number: int = Field(description="Approximate line number")
    severity: str = Field(description="critical, warning, info")
    description: str
    fix_suggestion: str

class CodeReview(BaseModel):
    bugs: list[Bug]
    overall_quality: int = Field(description="1-10 code quality score")
    improvements: list[str]

reviewer = llm.with_structured_output(CodeReview)

for name, sample in code_samples.items():
    print(f"\nReviewing: {name}")
    review = reviewer.invoke(f"Review this code for bugs and issues:\n```python\n{sample}\n```")
    print(f"  Quality: {review.overall_quality}/10")
    for bug in review.bugs:
        print(f"  [{bug.severity}] Line ~{bug.line_number}: {bug.description}")
        print(f"    Fix: {bug.fix_suggestion}")
    for imp in review.improvements:
        print(f"  Suggestion: {imp}")

## Step 4 — Code Refactoring Suggestions

In [ ]:
refactor_prompt = ChatPromptTemplate.from_messages([
    ("system", """Suggest how to refactor this code. Provide:
1. The refactored code
2. What changed and why
3. Any Pythonic improvements (list comprehensions, generators, etc.)
4. Error handling additions needed"""),
    ("human", "Refactor this code:\n```python\n{code}\n```")
])

refactor_chain = refactor_prompt | llm | StrOutputParser()
print("Refactoring suggestions for 'buggy_function':")
result = refactor_chain.invoke({"code": code_samples["buggy_function"]})
print(result)

## Step 5 — Complexity Analysis

In [ ]:
complexity_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the time and space complexity of this code.
Provide:
- Big-O time complexity with explanation
- Big-O space complexity
- Best/worst/average case analysis
- Suggest optimization if complexity can be improved"""),
    ("human", "Analyze complexity:\n```python\n{code}\n```")
])

complexity_chain = complexity_prompt | llm | StrOutputParser()

for name, sample in code_samples.items():
    print(f"\n{'='*60}")
    print(f"Complexity: {name}")
    print('='*60)
    analysis = complexity_chain.invoke({"code": sample})
    print(analysis)

## What You Learned
- **Line-by-line code explanation** for any snippet
- **Automated bug detection** with structured output
- **Refactoring suggestions** with Pythonic improvements
- **Complexity analysis** with Big-O notation